The chart above describes arrival delays for two airlines across five destinations. Your task is to
(1) Create a .CSV file (or optionally, a MySQL database!) that includes all of the information below. You may change
the structure of the information as you like.
(2) Read the information from your .CSV file into pandas, and perform analysis to compare the arrival delays for the
two airlines.
(3) Your code should be in a Jupyter (IPython) notebook (in GitHub), and should include narrative descriptions of your
analysis and conclusions.

In [118]:
import pandas as pd

df = pd.read_csv('flights.csv')

df

,Unnamed: 0,Unnamed: 1,Los Angeles,Phoenix,San Diego,San Franciso,Seattle
0,ALASKA,on time,497.0,221.0,212.0,503.0,1841.0
1,NaN,delayed,62.0,12.0,20.0,102.0,305.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AM WEST,on time,694.0,4840.0,383.0,320.0,201.0
4,NaN,delayed,117.0,415.0,65.0,129.0,61.0


## Importing CSV file into Dataframe
`df = pd.read_csv('flights.csv')`
- Reads the CSV file and turns it into a dataframe we can manipulate

In [119]:
df = df.dropna(how="all")
df.rename(columns={"Unnamed: 0": "Airline", "Unnamed: 1": "Status"}, inplace=True)

df

,Airline,Status,Los Angeles,Phoenix,San Diego,San Franciso,Seattle
0,ALASKA,on time,497.0,221.0,212.0,503.0,1841.0
1,NaN,delayed,62.0,12.0,20.0,102.0,305.0
3,AM WEST,on time,694.0,4840.0,383.0,320.0,201.0
4,NaN,delayed,117.0,415.0,65.0,129.0,61.0


## Cleaning NaN Rows & Headers

`df = df.dropna(how="all")`
* Gets rid of NaN rows, 
    * `(how="all")` Makes sure "all" cells in the row is NaN then deletes the row.
    * `(how="any")` If there is "any" value in the row with NaN it will delete the row.

`df.rename(columns={"Unnamed: 0": "Airline", "Unnamed: 1": "Status"}, inplace=True)`
* Will change the column header of the `Unnamed: 0` to `Airline` and `Unnamed: 1` to `Status`.


In [120]:
df = df.ffill(axis=0)

df

,Airline,Status,Los Angeles,Phoenix,San Diego,San Franciso,Seattle
0,ALASKA,on time,497.0,221.0,212.0,503.0,1841.0
1,ALASKA,delayed,62.0,12.0,20.0,102.0,305.0
3,AM WEST,on time,694.0,4840.0,383.0,320.0,201.0
4,AM WEST,delayed,117.0,415.0,65.0,129.0,61.0


## Filling in NaN value

`df.ffill(axis=0)`
* Known as Forward Fill will replace the NaN value of the with the previous row value.
    * `(axis=0)` deteremines which column
    * Will replace the NaN value cells in the first column with Alaska and AM West

In [121]:
df_melt = df.melt(id_vars=["Airline","Status"], var_name="City", value_name="Flights")
df_melt

,Airline,Status,City,Flights
0,ALASKA,on time,Los Angeles,497.0
1,ALASKA,delayed,Los Angeles,62.0
2,AM WEST,on time,Los Angeles,694.0
3,AM WEST,delayed,Los Angeles,117.0
4,ALASKA,on time,Phoenix,221.0
5,ALASKA,delayed,Phoenix,12.0
6,AM WEST,on time,Phoenix,4840.0
7,AM WEST,delayed,Phoenix,415.0
8,ALASKA,on time,San Diego,212.0
9,ALASKA,delayed,San Diego,20.0


## Melting the data
`id_vars=["Airline", "Status"]`
* Identifier variables - Keeps theses columns unchanged and keeps them as anchor columns.
`var_name="City"`   
* Took the rest of the header values and put them under the column of City.
`value_name="Flights"`
* Took all the data values found inside of the cells and put them under column of Flights.

In [122]:
city_airline = df_melt.pivot_table(
    index=['City', 'Airline'], 
    columns='Status', 
    values='Flights'
).reset_index()

city_airline['Total'] = city_airline['delayed'] + city_airline['on time']
city_airline['Percent Delayed'] = (city_airline['delayed'] / city_airline['Total']) * 100

city_airline

Status,City,Airline,delayed,on time,Total,Percent Delayed
0,Los Angeles,ALASKA,62.0,497.0,559.0,11.091234
1,Los Angeles,AM WEST,117.0,694.0,811.0,14.426634
2,Phoenix,ALASKA,12.0,221.0,233.0,5.150215
3,Phoenix,AM WEST,415.0,4840.0,5255.0,7.897241
4,San Diego,ALASKA,20.0,212.0,232.0,8.620690
5,San Diego,AM WEST,65.0,383.0,448.0,14.508929
6,San Franciso,ALASKA,102.0,503.0,605.0,16.859504
7,San Franciso,AM WEST,129.0,320.0,449.0,28.730512
8,Seattle,ALASKA,305.0,1841.0,2146.0,14.212488
9,Seattle,AM WEST,61.0,201.0,262.0,23.282443


## Pivot and Calculations

`df_melt.pivot_table`
* `index` - Chooses which column (or columns) will become your row labels along the left side.
* `columns` - Reads the status columns and split the `on time` and `delayed` into their own columns.
* `values` - Assigns the `flights` columns as its data values.
* `aggfunc` - Pandas to add up any matching flight numbers that fall into the same cell so they display as total counts rather than averages or mean.
* `.reset_index()` - Takes our columns from index row and pushes them back into standard data column.


`city_airline['Total'] = city_airline['delayed'] + city_airline['on time']`
* Adds the number of `delayed` and `on time` into the total.
`city_airline['Percent Delayed'] = (city_airline['delayed'] / city_airline['Total']) * 100`
* Calculates the percentage delayed for that city.